In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, random, os, json, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from pathlib import Path

selected_master_projects = [
    'Dubai Sports City', 'Silicon Oasis', 'Palm Jumeirah', 'Jumeirah Village Circle',
    'Jumeirah Lakes Towers', 'International City Phase 1', 'The Greens',
    'Dubai Marina', 'DownTown Dubai'
]

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
torch.use_deterministic_algorithms(True)
torch.set_float32_matmul_precision('medium')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

df_w      = pd.read_csv("../1-transaction-data/price_series.csv",   index_col="date", parse_dates=True)
counts_df = pd.read_csv("../1-transaction-data/counts_series.csv", index_col="date", parse_dates=True)

window_options = [6, 8, 10, 12, 14, 16]
horizons       = [16, 20, 24, 28, 32]
region_targets = selected_master_projects + ['Global']
cutoff         = pd.Timestamp('2024-01-01')
tags_short     = ['P', 'PC']

# data prep 
def build_flat(tag, h, tgt, win):
    X, y, d = [], [], []
    for i in range(win, len(df_w) - h):
        d.append(df_w.index[i])
        p = df_w[tgt].iloc[i-win:i].values if 'P' in tag else np.empty(0)
        c = counts_df[tgt].iloc[i-win:i].values if 'C' in tag else np.empty(0)
        X.append(np.hstack([p, c])); y.append(df_w[tgt].iloc[i + h])
    return np.vstack(X), np.array(y), pd.DatetimeIndex(d)

def build_seq(tag, h, tgt, win):
    X, y, d = [], [], []
    for i in range(win, len(df_w) - h):
        d.append(df_w.index[i])
        p = df_w[tgt].iloc[i-win:i].values.reshape(-1,1) if 'P' in tag else np.empty((win,0))
        c = counts_df[tgt].iloc[i-win:i].values.reshape(-1,1) if 'C' in tag else np.empty((win,0))
        X.append(np.hstack([p, c])); y.append(df_w[tgt].iloc[i + h])
    return np.stack(X), np.array(y), pd.DatetimeIndex(d)


class LSTMNet(nn.Module):
    def __init__(self, ch, h, num_layers, dropout):
        super().__init__()
        self.inp  = nn.Linear(ch, h)
        self.rnn  = nn.LSTM(h, h, num_layers=num_layers, batch_first=True,
                            dropout=dropout, bidirectional=True)
        self.head = nn.Sequential(nn.LayerNorm(h*2),
                                  nn.Linear(h*2, h*2), nn.ReLU(),
                                  nn.Linear(h*2, 1))
    def forward(self, x):
        return self.head(self.rnn(self.inp(x))[0][:, -1])

def fit_lstm(Xtr, ytr, Xval, ch, h, num_layers, dropout, lr):
    sx = StandardScaler().fit(Xtr.reshape(-1, ch))
    sy = StandardScaler().fit(ytr.reshape(-1, 1))
    Xtr_s = sx.transform(Xtr.reshape(-1, ch)).reshape(Xtr.shape)
    Xval_s = sx.transform(Xval.reshape(-1, ch)).reshape(Xval.shape)
    ytr_s = sy.transform(ytr.reshape(-1, 1)).ravel()
    n_val_int = max(1, int(0.10 * len(Xtr_s)))
    X_train, X_int = Xtr_s[:-n_val_int], Xtr_s[-n_val_int:]
    y_train, y_int = ytr_s[:-n_val_int], ytr_s[-n_val_int:]

    ld_train = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                        torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)),
                          batch_size=128, shuffle=False)
    ld_int   = DataLoader(TensorDataset(torch.tensor(X_int, dtype=torch.float32),
                                        torch.tensor(y_int, dtype=torch.float32).unsqueeze(1)),
                          batch_size=128, shuffle=False)

    mdl = LSTMNet(ch, h, num_layers, dropout).to(device)
    opt = torch.optim.AdamW(mdl.parameters(), lr, weight_decay=1e-4)
    best_loss, patience_left, best_state = 1e9, 5, None
    for _ in range(60):
        mdl.train()
        for xb, yb in ld_train:
            opt.zero_grad()
            loss = nn.functional.mse_loss(mdl(xb.to(device)), yb.to(device))
            loss.backward(); nn.utils.clip_grad_norm_(mdl.parameters(), 1.0); opt.step()
        mdl.eval()
        with torch.no_grad():
            vpred = torch.cat([mdl(xb.to(device)) for xb, _ in ld_int]).cpu().squeeze(1)
            vloss = nn.functional.mse_loss(vpred, torch.tensor(y_int, dtype=torch.float32))
        if vloss < best_loss:
            best_loss, patience_left, best_state = vloss.item(), 5, mdl.state_dict()
        else:
            patience_left -= 1
            if patience_left == 0: break
    mdl.load_state_dict(best_state); mdl.eval()
    with torch.no_grad():
        pr_s = mdl(torch.tensor(Xval_s, dtype=torch.float32).to(device)).cpu().squeeze(1).numpy()
    return sy.inverse_transform(pr_s.reshape(-1,1)).ravel()

# search spaces
param_grids = {
    'Ridge': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
              'fit_intercept': [True, False]},
    'RF':    {'n_estimators': [100, 200, 300],
              'max_depth': [None, 5, 8, 12],
              'min_samples_leaf': [1, 2, 4]},
    'KNN':   {'n_neighbors': [3, 5, 7, 9],
              'weights': ['uniform', 'distance'],
              'leaf_size': [30, 50],
              'p': [1, 2]},
    'LSTM':  {'h': [32, 64, 128],
              'num_layers': [1, 2, 3],
              'dropout': [0.0, 0.2, 0.4],
              'lr': [1e-3, 3e-4, 1e-4]}
}

best_params   = {m: None for m in param_grids}
best_scores   = {m: 1e9  for m in param_grids}
trial_history = []

total_trials  = len(param_grids) * 30
trial_counter = 0

# hyper‑parameter search 
for model_name, grid in param_grids.items():
    for trial in range(30):
        cfg = {k: random.choice(v) for k, v in grid.items()}
        cfg['window'] = random.choice(window_options)
        mae_sum, cnt = 0.0, 0

        for tag in tags_short:
            for reg in region_targets:
                for h in horizons:
                    if model_name == 'LSTM':
                        X, y, dts = build_seq(tag, h, reg, cfg['window'])
                    else:
                        X, y, dts = build_flat(tag, h, reg, cfg['window'])

                    mask_train = dts < cutoff
                    X_train_all, y_train_all = X[mask_train], y[mask_train]
                    if len(y_train_all) < 10:
                        continue

                    n_val = max(1, int(0.20 * len(X_train_all)))
                    X_train, X_val = X_train_all[:-n_val], X_train_all[-n_val:]
                    y_train, y_val = y_train_all[:-n_val], y_train_all[-n_val:]

                    if model_name == 'Ridge':
                        mdl = Ridge(alpha=cfg['alpha'], fit_intercept=cfg['fit_intercept'])
                        pred = mdl.fit(X_train, y_train).predict(X_val)
                    elif model_name == 'RF':
                        mdl = RandomForestRegressor(n_estimators=cfg['n_estimators'],
                                                    max_depth=cfg['max_depth'],
                                                    min_samples_leaf=cfg['min_samples_leaf'],
                                                    random_state=SEED, n_jobs=-1)
                        pred = mdl.fit(X_train, y_train).predict(X_val)
                    elif model_name == 'KNN':
                        mdl = Pipeline([('sc', StandardScaler()),
                                        ('knn', KNeighborsRegressor(
                                            n_neighbors=cfg['n_neighbors'],
                                            weights=cfg['weights'],
                                            leaf_size=cfg['leaf_size'],
                                            p=cfg['p'],
                                            n_jobs=-1))])
                        pred = mdl.fit(X_train, y_train).predict(X_val)
                    else:
                        pred = fit_lstm(X_train, y_train, X_val, X.shape[2],
                                        cfg['h'], cfg['num_layers'],
                                        cfg['dropout'], cfg['lr'])
                    mae_sum += mean_absolute_error(y_val, pred)
                    cnt += 1

        if cnt == 0:
            continue
        val_mae = mae_sum / cnt
        trial_history.append({'model': model_name, 'params': cfg, 'val_mae': val_mae})

        if val_mae < best_scores[model_name]:
            best_scores[model_name] = val_mae
            best_params[model_name] = cfg

        trial_counter += 1
        if trial_counter % 2 == 0 or trial_counter == total_trials:
            print(f'Progress: completed {trial_counter} / {total_trials} trials')

    print(f'{model_name} search finished – best val MAE: {best_scores[model_name]:.4f}')

# global best window
window_perf = {}
for m, cfg in best_params.items():
    w = cfg['window']
    window_perf.setdefault(w, []).append(best_scores[m])
best_window = min(window_perf.items(), key=lambda x: np.mean(x[1]))[0]

Path('param_search_results').mkdir(exist_ok=True)
with open('param_search_results/param_opt_results.json', 'w') as f:
    json.dump({
        'best_params' : best_params,
        'best_val_mae': best_scores,
        'best_window' : best_window,
        'trials'      : trial_history
    }, f, indent=2)

print('Best parameters per model (validation MAE)')
for m in best_params:
    print(m, best_params[m], 'MAE', best_scores[m])
print('Global best window length', best_window)


Running on: cuda
Progress: completed 2 / 120 trials
Progress: completed 4 / 120 trials
Progress: completed 6 / 120 trials
Progress: completed 8 / 120 trials
Progress: completed 10 / 120 trials
Progress: completed 12 / 120 trials
Progress: completed 14 / 120 trials
Progress: completed 16 / 120 trials
Progress: completed 18 / 120 trials
Progress: completed 20 / 120 trials
Progress: completed 22 / 120 trials
Progress: completed 24 / 120 trials
Progress: completed 26 / 120 trials
Progress: completed 28 / 120 trials
Progress: completed 30 / 120 trials
Ridge search finished – best val MAE: 6.2301
Progress: completed 32 / 120 trials
Progress: completed 34 / 120 trials
Progress: completed 36 / 120 trials
Progress: completed 38 / 120 trials
Progress: completed 40 / 120 trials
Progress: completed 42 / 120 trials
Progress: completed 44 / 120 trials
Progress: completed 46 / 120 trials
Progress: completed 48 / 120 trials
Progress: completed 50 / 120 trials
Progress: completed 52 / 120 trials
Progre

c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warning

Progress: completed 102 / 120 trials
Progress: completed 104 / 120 trials
Progress: completed 106 / 120 trials


c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warning

Progress: completed 108 / 120 trials


...

Progress: completed 112 / 120 trials
Progress: completed 114 / 120 trials


c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warning

Progress: completed 116 / 120 trials


c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warnings.warn(
c:\venv\realestate\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.4 and num_layers=1
  warning

Progress: completed 118 / 120 trials
Progress: completed 120 / 120 trials
LSTM search finished – best val MAE: 6.9529
Best parameters per model (validation MAE)
Ridge {'alpha': 10.0, 'fit_intercept': False, 'window': 6} MAE 6.230145933456556
RF {'n_estimators': 300, 'max_depth': 5, 'min_samples_leaf': 1, 'window': 12} MAE 6.6235549727130545
KNN {'n_neighbors': 7, 'weights': 'uniform', 'leaf_size': 30, 'p': 1, 'window': 16} MAE 6.657802762612864
LSTM {'h': 32, 'num_layers': 3, 'dropout': 0.0, 'lr': 0.0001, 'window': 12} MAE 6.952869384522988
Global best window length 6


In [ ]:
# initial run results

# Best parameters per model (validation MAE)
# Ridge {'alpha': 10.0, 'fit_intercept': False, 'window': 6} MAE 7.476265167227478
# RF {'n_estimators': 300, 'max_depth': 5, 'min_samples_leaf': 1, 'window': 12} MAE 9.13617437654979
# KNN {'n_neighbors': 3, 'weights': 'uniform', 'leaf_size': 30, 'p': 1, 'window': 14} MAE 9.3019707650267
# LSTM {'h': 32, 'num_layers': 2, 'dropout': 0.0, 'lr': 0.0003, 'window': 14} MAE 9.46951258650024
# Global best window length 6